In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/xndung/fidgety/data")
MODE = "paper"  # "paper" or "smoke"
SEED = 42
OUTPUT_DIR = Path("/kaggle/working/outputs/paper_v1")
RESUME = True

if MODE == "paper":
    FOLDS = list(range(1, 10))
    ARCHITECTURES = list(range(1, 11))
    RUNS = 20
    EPOCHS = 1000
elif MODE == "smoke":
    FOLDS = [1]
    ARCHITECTURES = [1]
    RUNS = 2
    EPOCHS = 5
else:
    raise ValueError("Choose paper or smoke.")


In [ ]:
import subprocess
import sys

SOURCE_DIR = Path("/kaggle/working/Fidgety")
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
if SOURCE_DIR.exists():
    subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(SOURCE_DIR / "requirements.txt")], check=True)
REVISION = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source:", REVISION)


In [ ]:
import json
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Settings before training.")
if not (DATA_ROOT / "data_mat_9fold").is_dir() and (DATA_ROOT / "data" / "data_mat_9fold").is_dir():
    DATA_ROOT = DATA_ROOT / "data"
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(DATA_ROOT)

CONFIG = dict(mode=MODE, data_root=str(DATA_ROOT.resolve()), folds=FOLDS,
              architectures=ARCHITECTURES, runs=RUNS, epochs=EPOCHS, seed=SEED,
              source_commit=REVISION)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_DIR / "config.json"
if CONFIG_PATH.exists():
    if not RESUME:
        raise FileExistsError("Choose a new OUTPUT_DIR or set RESUME=True.")
    if json.loads(CONFIG_PATH.read_text()) != CONFIG:
        raise ValueError("Existing experiment has a different config/source. Choose a new OUTPUT_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
CSV_PATH = OUTPUT_DIR / "results.csv"
print("GPU:", torch.cuda.get_device_name(0))
print("Mode:", MODE)
print("Architectures:", len(ARCHITECTURES), "per modality; modalities: MAT, IMU, VID, ALL 1-Net")
print("Training jobs:", 4 * len(FOLDS) * len(ARCHITECTURES))
print("Total restarts:", 4 * len(FOLDS) * len(ARCHITECTURES) * RUNS)
print("Max epochs:", EPOCHS, "| batch size: 4 | validation split: 1/8 | patience: 10")
print("Architectures: published Supplementary Tables 2 and 3; no hyperparameter search is rerun.")
print("Late fusion: mean probabilities, not mean logits or voting.")
print("Main comparison: published architecture #1 for each modality, fixed before this retraining.")
print("All 10 architectures are also reported individually; no architecture is selected using new test scores.")
print("Full paper mode: 7200 training restarts; completed jobs can be skipped on rerun.")
print("Interrupted jobs restart their restarts; epoch/optimizer resume is not implemented.")
print("CSV:", CSV_PATH)

import pandas as pd
from IPython.display import display

catalog = json.loads((SOURCE_DIR / "paper_architectures.json").read_text())["models"]
shapes = {"mat": (500, 6), "imu": (300, 36), "vid": (250, 60), "fusion": (250, 102)}
architecture_rows = []
for modality, published in catalog.items():
    for architecture in ARCHITECTURES:
        p = published[str(architecture)]
        frames, features = shapes[modality]
        parameters = p["kn1"] * (p["kw1"] + 1)
        parameters += p["kn2"] * (p["kn1"] * p["kw2"] + 1)
        parameters += p["kn3"] * (p["kn2"] * p["kw3"] + 1)
        parameters += p["fc1"] * (features * p["kn3"] + 1) + p["fc1"] + 1
        parameters += 2 * (p["kn1"] + p["kn2"] + p["kn3"] + p["fc1"])
        architecture_rows.append(dict(modality=modality, architecture=architecture,
                                      input_shape=f"{frames}x{features}", **p,
                                      pooling=f"{frames}x1", flatten_features=features * p["kn3"],
                                      trainable_parameters=parameters))
print("Published architectures used in this experiment:")
display(pd.DataFrame(architecture_rows))
print("Layer order: [Conv-ReLU-BatchNorm-Dropout(0.2)] x3 -> temporal average pool -> channels-last flatten -> Dense-ReLU-BatchNorm-Dropout(0.2) -> linear logit.")


In [ ]:
env = os.environ.copy()
env.pop("FIDGETY_DATA_ZIP", None)
env["PYTHONUNBUFFERED"] = "1"
with subprocess.Popen([sys.executable, "-u", str(SOURCE_DIR / "paper_runner.py"), "--config", str(CONFIG_PATH)],
                      env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if returncode:
    raise RuntimeError("Training failed. Inspect the printed error and per-job train.log. Completed jobs remain saved.")


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

if not CSV_PATH.exists():
    raise FileNotFoundError("No completed job yet: " + str(CSV_PATH))
results = pd.read_csv(CSV_PATH)
complete = bool(results["full_paper_schedule_complete"].iloc[0])
print("FULL PAPER TRAINING SCHEDULE COMPLETE" if complete else "SMOKE OR PARTIAL RESULTS - NOT A PAPER REPRODUCTION")
print("Completed jobs:", int(results["completed_training_jobs"].iloc[0]), "/", int(results["expected_training_jobs"].iloc[0]))
print("Each reported fold uses its lowest-validation-loss restart; restarts are not averaged.")
print("CI95: mean +/- 1.96 * sample_std / sqrt(number_of_folds), as in the author's MATLAB code.")
print("Numbers are percentages. delta_pp is this run minus the published value, in percentage points.")
print("Main models use published architecture #1, not a new ranking by test-set performance.")

order = ["MAT", "IMU", "VID", "MAT+IMU", "MAT+VID", "IMU+VID", "ALL 3-Nets", "ALL 1-Net"]
table2 = results[results.row_type == "table2"].copy()
table2["condition"] = pd.Categorical(table2["condition"], categories=order, ordered=True)
table2 = table2.sort_values("condition")
columns = ["condition", "n_folds", "sensitivity_mean_pct", "sensitivity_ci_low_pct", "sensitivity_ci_high_pct",
           "specificity_mean_pct", "specificity_ci_low_pct", "specificity_ci_high_pct",
           "balanced_accuracy_mean_pct", "balanced_accuracy_ci_low_pct", "balanced_accuracy_ci_high_pct",
           "balanced_accuracy_paper_pct", "balanced_accuracy_delta_pp"]
print("Table 2 comparison + ALL 1-Net:")
display(table2[columns].round(3))

pairs = results[results.row_type == "table3"]
print("Table 3: paired, two-sided exact Wilcoxon signed-rank p-values (uncorrected).")
if len(pairs):
    matrix = pd.DataFrame(np.nan, index=order, columns=order)
    for row in pairs.itertuples():
        matrix.loc[row.condition_a, row.condition_b] = row.p_value
        matrix.loc[row.condition_b, row.condition_a] = row.p_value
    display(matrix.round(5))
else:
    print("At least two completed folds are required; smoke mode does not produce inferential comparisons.")

print("Supplementary Table 4 / 5 structure: every published architecture, each summarized across folds.")
architecture_summary = results[results.row_type == "architecture_summary"]
display(architecture_summary[["condition", "architecture", "n_folds", "sensitivity_mean_pct",
                              "specificity_mean_pct", "balanced_accuracy_mean_pct",
                              "balanced_accuracy_ci_low_pct", "balanced_accuracy_ci_high_pct"]].round(3))
print("Fold results and selected restart IDs:")
display(results[results.row_type == "fold"][["condition", "architecture", "fold", "selected_run",
                                          "best_val_loss", "sensitivity", "specificity", "balanced_accuracy"]])
print("CSV:", CSV_PATH)
display(FileLink(str(CSV_PATH)))
print("Source: https://doi.org/10.1038/s43856-024-00701-w")
print("Reference numbers / CI method: https://zenodo.org/records/14046527")
print("PyTorch retraining is not numerically identical to TensorFlow 2.10; matching the table format does not validate the model by itself.")


In [ ]:
import matplotlib.pyplot as plt

if len(table2):
    positions = np.arange(len(table2))
    values = table2["balanced_accuracy_mean_pct"].to_numpy(dtype=float)
    margin = (table2["balanced_accuracy_ci_high_pct"] - table2["balanced_accuracy_mean_pct"]).to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(positions, values, facecolor="none", edgecolor="tab:blue", label="PyTorch retraining")
    finite = np.isfinite(margin)
    if finite.any():
        ax.errorbar(positions[finite], values[finite], yerr=margin[finite], fmt="none", capsize=4, color="tab:red", label="95% CI")
    ax.scatter(positions, table2["balanced_accuracy_paper_pct"], marker="D", color="black", label="Published reference")
    ax.set_xticks(positions, table2["condition"].astype(str), rotation=20)
    ax.set_ylabel("Balanced accuracy (%)")
    ax.set_ylim(0, 100)
    ax.set_title("Paper comparison" if complete else "Smoke / partial comparison")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "paper_comparison.png", dpi=180)
    plt.show()
